# Advanced Phishing Detection

# 1. Imports & Configuration

In [2]:
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded. RANDOM_STATE=', RANDOM_STATE)

ModuleNotFoundError: No module named 'pandas'

# 2. Load Dataset & Label Normalization

In [ ]:
df = pd.read_csv('phishing_dataset.csv')
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head(3))

def label_to_int(x):
    s = str(x).strip().lower()
    return 1 if s in ['bad','phishing','malicious','1','true','yes'] else 0

df['Label_numeric'] = df['Label'].apply(label_to_int)
print('Class balance:', df['Label_numeric'].value_counts(normalize=True))
assert 'URL' in df.columns, 'Expected column URL not found'
assert 'Label_numeric' in df.columns, 'Label mapping failed'

# 3. Numeric URL Feature Engineering & Splitting

In [ ]:
def extract_url_numeric_features(url: str) -> dict:
    if not isinstance(url, str):
        url = ''
    try:
        parsed = urlparse(url if re.match(r'^https?://', url) else 'http://' + url)
        domain = parsed.netloc or ''
        path = parsed.path or ''
        query = parsed.query or ''
        s = url
        feats = {}
        
        feats['url_len'] = len(s)
        feats['num_digits'] = sum(ch.isdigit() for ch in s)
        feats['num_letters'] = sum(ch.isalpha() for ch in s)
        feats['num_special'] = sum((not ch.isalnum()) and (not ch.isspace()) for ch in s)
       
        for ch, name in [('.', 'dots'),('-', 'hyphens'),('_','underscores'),('/', 'slashes'),('?', 'qmarks'),('=','equals'),('&','amps'),('%','perc'),('#','hash')]:
            feats[f'cnt_{name}'] = s.count(ch)
       
        feats['domain_len'] = len(domain)
        feats['path_len'] = len(path)
        feats['path_depth'] = path.count('/')
        feats['query_len'] = len(query)
        feats['num_params'] = (query.count('&') + 1) if query else 0
        feats['has_https'] = 1 if parsed.scheme == 'https' else 0
        feats['has_www'] = 1 if domain.startswith('www.') else 0
       
        feats['domain_is_ip'] = 1 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', domain) else 0
        feats['suspicious_tld'] = 1 if re.search(r'\.(tk|ml|ga|cf|gq|xyz|top|ru|click|link|bid|party|win)(?:/|$)', domain.lower()) else 0
        feats['brand_terms'] = 1 if re.search(r'(paypal|google|facebook|apple|microsoft|yahoo|amazon)', s.lower()) else 0
        feats['num_subdomains'] = max(0, domain.count('.') - 1) if domain else 0

        # Additional ratios
        feats['digits_ratio'] = feats['num_digits'] / max(1, feats['url_len'])
        feats['special_ratio'] = feats['num_special'] / max(1, feats['url_len'])
        feats['letters_ratio'] = feats['num_letters'] / max(1, feats['url_len'])
        feats['hyphen_domain_ratio'] = feats['cnt_hyphens'] / max(1, feats['domain_len'])

        # Entropy features for domain and path
        def _entropy(text: str) -> float:
            if not text:
                return 0.0
            from collections import Counter
            import math
            c = Counter(text)
            p = [v/len(text) for v in c.values()]
            return -sum(pi * math.log2(pi) for pi in p if pi > 0)

        feats['domain_entropy'] = _entropy(domain)
        feats['path_entropy'] = _entropy(path)

        return feats
    except Exception:
        return {
            'url_len':0,'num_digits':0,'num_letters':0,'num_special':0,
            'cnt_dots':0,'cnt_hyphens':0,'cnt_underscores':0,'cnt_slashes':0,'cnt_qmarks':0,'cnt_equals':0,'cnt_amps':0,'cnt_perc':0,'cnt_hash':0,
            'domain_len':0,'path_len':0,'path_depth':0,'query_len':0,'num_params':0,'has_https':0,'has_www':0,'domain_is_ip':0,'suspicious_tld':0,'brand_terms':0,'num_subdomains':0,
            'digits_ratio':0.0,'special_ratio':0.0,'letters_ratio':0.0,'hyphen_domain_ratio':0.0,
            'domain_entropy':0.0,'path_entropy':0.0,
        }


def build_numeric_feature_frame(url_series: pd.Series) -> pd.DataFrame:
    rows = [extract_url_numeric_features(u) for u in url_series.tolist()]
    return pd.DataFrame(rows)

X_num = build_numeric_feature_frame(df['URL'])
y = df['Label_numeric'].astype(int).values
print('Numeric features shape:', X_num.shape)


X_train_num, X_test_num, y_train, y_test, train_urls, test_urls = train_test_split(
    X_num, y, df['URL'].values, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_num, X_val_num, y_train, y_val, train_urls, val_urls = train_test_split(
    X_train_num, y_train, train_urls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)
print('Train/Val/Test sizes:', len(y_train), len(y_val), len(y_test))

In [ ]:
use_transformer = False
st_model = None
vectorizer = None

try:
    from sentence_transformers import SentenceTransformer
    model_name = 'all-MiniLM-L6-v2'
    st_model = SentenceTransformer(model_name)
    use_transformer = True
    print('Using SentenceTransformer:', model_name)
except Exception as e:
    print('Transformer not available, fallback to TF-IDF char n-grams. Reason:', e)
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2, max_features=50000)

def fit_text_features(train_url_list):
    if use_transformer:
        emb = st_model.encode(list(train_url_list), convert_to_numpy=True, show_progress_bar=False)
        return emb, None
    else:
        X = vectorizer.fit_transform(train_url_list)
        return X, vectorizer

def transform_text_features(url_list, vec=None):
    if use_transformer:
        return st_model.encode(list(url_list), convert_to_numpy=True, show_progress_bar=False)
    else:
        return vec.transform(url_list)

# Fit on train URLs
X_train_text, fitted_vec = fit_text_features(train_urls)
X_val_text = transform_text_features(val_urls, fitted_vec)
X_test_text = transform_text_features(test_urls, fitted_vec)

print('Text features shapes:', getattr(X_train_text, 'shape', None), getattr(X_val_text, 'shape', None), getattr(X_test_text, 'shape', None))

# 5. Scaling & Base Model Training

In [ ]:
scaler_num = StandardScaler()
X_train_num_scaled = scaler_num.fit_transform(X_train_num)
X_val_num_scaled = scaler_num.transform(X_val_num)
X_test_num_scaled = scaler_num.transform(X_test_num)
print('Numeric scaling done.')

models = {}
metrics = {}

from collections import Counter
pos = Counter(y_train)[1]
neg = Counter(y_train)[0]
scale_pos_weight = max(1.0, neg / max(1, pos))


def eval_and_store(name, model, Xtr, ytr, Xva, yva):
    t0 = time.time()
    model.fit(Xtr, ytr)
    t1 = time.time()
    yva_pred = model.predict(Xva)
    acc = accuracy_score(yva, yva_pred)
    prec = precision_score(yva, yva_pred)
    rec = recall_score(yva, yva_pred)
    f1 = f1_score(yva, yva_pred)
    proba = None
    if hasattr(model, 'predict_proba'):
        try:
            proba = model.predict_proba(Xva)[:,1]
        except Exception:
            proba = None
    models[name] = model
    metrics[name] = {'val_accuracy': acc, 'val_precision': prec, 'val_recall': rec, 'val_f1': f1, 'train_time_s': t1-t0, 'has_proba': proba is not None}
    print(f"{name}: acc={acc:.4f}, f1={f1:.4f}, time={t1-t0:.2f}s")
    return proba

# 1) XGBoost (primary) on numeric features with early stopping and class weighting
xgb = XGBClassifier(
    n_estimators=2000, learning_rate=0.03, max_depth=6, subsample=0.9, colsample_bytree=0.8,
    min_child_weight=1, reg_lambda=1.0,
    random_state=RANDOM_STATE, eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)
# Fit with early stopping
xgb.fit(
    X_train_num, y_train,
    eval_set=[(X_val_num, y_val)],
    verbose=False,
    early_stopping_rounds=100
)
# Evaluate/record with same helper for consistency
proba_xgb = eval_and_store('XGBoost', xgb, X_train_num, y_train, X_val_num, y_val)

# 2) Transformer/TF-IDF + Logistic Regression on text features
lr_text = LogisticRegression(max_iter=1000, C=2.0, class_weight='balanced', random_state=RANDOM_STATE)
proba_text = eval_and_store('Transformer', lr_text, X_train_text, y_train, X_val_text, y_val)

# 3) Gradient Boosting on numeric features
gb = GradientBoostingClassifier(n_estimators=400, learning_rate=0.05, max_depth=3, subsample=0.9, random_state=RANDOM_STATE)
proba_gb = eval_and_store('GradientBoosting', gb, X_train_num, y_train, X_val_num, y_val)

# 4) Logistic Regression (numeric features, scaled)
lr_num = LogisticRegression(max_iter=1000, C=1.5, class_weight='balanced', random_state=RANDOM_STATE)
proba_lr = eval_and_store('LogisticRegression', lr_num, X_train_num_scaled, y_train, X_val_num_scaled, y_val)

print('\nValidation metrics:')
pd.DataFrame(metrics).T

# Utility: find best threshold on validation
from sklearn.metrics import f1_score

def best_threshold(y_true, y_score, metric=f1_score):
    best_t, best_m = 0.5, -1.0
    for t in np.linspace(0.1, 0.9, 81):
        y_pred = (y_score >= t).astype(int)
        try:
            m = metric(y_true, y_pred)
        except Exception:
            m = 0.0
        if m > best_m:
            best_m, best_t = m, t
    return best_t, best_m

# 6. Fusion / Ensemble Strategies

In [ ]:
# Threshold tuning for ensembles

# Collect available validation probabilities
probas = {}
if 'XGBoost' in models and metrics['XGBoost']['has_proba']:
    probas['xgb'] = models['XGBoost'].predict_proba(X_val_num)[:,1]
if 'Transformer' in models and metrics['Transformer']['has_proba']:
    probas['text'] = models['Transformer'].predict_proba(X_val_text)[:,1]
if 'GradientBoosting' in models and metrics['GradientBoosting']['has_proba']:
    probas['gb'] = models['GradientBoosting'].predict_proba(X_val_num)[:,1]
if 'LogisticRegression' in models and metrics['LogisticRegression']['has_proba']:
    probas['lr'] = models['LogisticRegression'].predict_proba(X_val_num_scaled)[:,1]

if not probas:
    print('No probability-capable models available; skipping threshold tuning.')
else:
    # Simple weighted average (equal weights or learned weights if present)
    keys = sorted(probas.keys())
    P = np.vstack([probas[k] for k in keys]).T
    w = np.ones(P.shape[1]) / P.shape[1]
    p_avg = (P @ w)

    t_avg, f1_avg = best_threshold(y_val, p_avg)
    print(f"Weighted-average ensemble (equal weights): best_t={t_avg:.3f}, val_f1={f1_avg:.4f}")

    # Apply on test
    # Build corresponding test probas
    test_probas = []
    if 'xgb' in keys:
        test_probas.append(models['XGBoost'].predict_proba(X_test_num)[:,1])
    if 'text' in keys:
        test_probas.append(models['Transformer'].predict_proba(X_test_text)[:,1])
    if 'gb' in keys:
        test_probas.append(models['GradientBoosting'].predict_proba(X_test_num)[:,1])
    if 'lr' in keys:
        test_probas.append(models['LogisticRegression'].predict_proba(X_test_num_scaled)[:,1])
    P_test = np.vstack(test_probas).T
    p_avg_test = (P_test @ w)

    y_pred_test = (p_avg_test >= t_avg).astype(int)
    print('Test metrics (tuned-avg ensemble):',
          'acc=', accuracy_score(y_test, y_pred_test),
          'f1=', f1_score(y_test, y_pred_test),
          'prec=', precision_score(y_test, y_pred_test),
          'rec=', recall_score(y_test, y_pred_test))

In [ ]:
# Fusion strategies: weighted average and stacking (blending)
val_probas = []
names = []
for name, proba in [('XGBoost', proba_xgb), ('Transformer', proba_text), ('GradientBoosting', proba_gb), ('LogisticRegression', proba_lr)]:
    if proba is not None:
        val_probas.append(proba.reshape(-1,1))
        names.append(name)

if len(val_probas) == 0:
    raise RuntimeError('No probability outputs available for fusion.')

P_val = np.hstack(val_probas)
print('Stacked validation probas shape:', P_val.shape, 'models:', names)

# Weights based on validation F1
weights = np.array([metrics[n]['val_f1'] for n in names], dtype=float)
weights = weights / (weights.sum() + 1e-8)
print('Weights (by val F1):', dict(zip(names, weights.round(4))))

# Fit a meta-classifier on validation probas (blending)
meta = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta.fit(P_val, y_val)
print('Meta model trained on validation set.')

# Evaluate on the test set
# Gather test-set probas for available models (ensure order matches names)
test_probas = []
for n in names:
    if n == 'XGBoost':
        test_probas.append(models[n].predict_proba(X_test_num)[:,1].reshape(-1,1))
    elif n == 'Transformer':
        test_probas.append(models[n].predict_proba(X_test_text)[:,1].reshape(-1,1))
    elif n == 'GradientBoosting':
        test_probas.append(models[n].predict_proba(X_test_num)[:,1].reshape(-1,1))
    elif n == 'LogisticRegression':
        test_probas.append(models[n].predict_proba(X_test_num_scaled)[:,1].reshape(-1,1))

P_test = np.hstack(test_probas)

# Weighted average ensemble
avg_proba = (P_test * weights.reshape(1,-1)).sum(axis=1)
y_pred_avg = (avg_proba >= 0.5).astype(int)
acc_avg = accuracy_score(y_test, y_pred_avg)
f1_avg = f1_score(y_test, y_pred_avg)

# Stacking (meta on validation probas)
stack_proba = meta.predict_proba(P_test)[:,1]
y_pred_stack = (stack_proba >= 0.5).astype(int)
acc_stack = accuracy_score(y_test, y_pred_stack)
f1_stack = f1_score(y_test, y_pred_stack)

print(f'Weighted Ensemble - Test Accuracy: {acc_avg:.4f}, F1: {f1_avg:.4f}')
print(f'Stacking Ensemble - Test Accuracy: {acc_stack:.4f}, F1: {f1_stack:.4f}')

# Also report best single model on test for reference
single_scores = {}
single_scores['XGBoost'] = accuracy_score(y_test, models['XGBoost'].predict(X_test_num))
single_scores['GradientBoosting'] = accuracy_score(y_test, models['GradientBoosting'].predict(X_test_num))
single_scores['LogisticRegression'] = accuracy_score(y_test, models['LogisticRegression'].predict(X_test_num_scaled))
single_scores['Transformer'] = accuracy_score(y_test, models['Transformer'].predict(X_test_text))
print('Single model test accuracies:', single_scores)

# Pick best by accuracy
best_name = 'WeightedEnsemble' if acc_avg >= acc_stack else 'StackingEnsemble'
best_acc = max(acc_avg, acc_stack)
best_pred = y_pred_avg if best_name=='WeightedEnsemble' else y_pred_stack

print(f'\nBest approach on test: {best_name} with Accuracy={best_acc:.4f}')
print('\nClassification report (best):')
print(classification_report(y_test, best_pred))
cm = confusion_matrix(y_test, best_pred)
print('Confusion matrix:\n', cm)

# 7. Save Artifact

In [ ]:
import joblib
artifact = {
    'models': models,
    'meta': meta,
    'weights': weights,
    'names': names,
    'scaler_num': scaler_num,
    'use_transformer': use_transformer,
    'fitted_vec': fitted_vec if not use_transformer else None,
}
joblib.dump(artifact, 'advanced_fusion_artifact.pkl')
print('Saved artifact -> advanced_fusion_artifact.pkl')

import joblib

def load_artifact(path='advanced_fusion_artifact.pkl'):
    return joblib.load(path)

def predict_url(url: str, artifact):
    # Build numeric features
    Xn = pd.DataFrame([extract_url_numeric_features(url)])
    Xn_scaled = artifact['scaler_num'].transform(Xn)
    # Build text features
    if artifact['use_transformer']:
        try:
            from sentence_transformers import SentenceTransformer
            st = SentenceTransformer('all-MiniLM-L6-v2')
            Xt = st.encode([url], convert_to_numpy=True, show_progress_bar=False)
        except Exception:
            # fallback TF-IDF if transformer unavailable at inference time
            vec = artifact['fitted_vec']
            Xt = vec.transform([url])
    else:
        vec = artifact['fitted_vec']
        Xt = vec.transform([url])
    # Collect probabilities in stored order
    probas = []
    for n in artifact['names']:
        if n == 'XGBoost':
            probas.append(artifact['models'][n].predict_proba(Xn)[:,1])
        elif n == 'GradientBoosting':
            probas.append(artifact['models'][n].predict_proba(Xn)[:,1])
        elif n == 'LogisticRegression':
            probas.append(artifact['models'][n].predict_proba(Xn_scaled)[:,1])
        elif n == 'Transformer':
            probas.append(artifact['models'][n].predict_proba(Xt)[:,1])
    P = np.vstack(probas).T
    # Weighted average by default
    avg_proba = float((P * artifact['weights'].reshape(1,-1)).sum(axis=1)[0])
    yhat = int(avg_proba >= 0.5)
    return {'prediction': yhat, 'probability': avg_proba}
